In [10]:
import sys; sys.path.append("..")          
import pandas as pd                          
import numpy as np                           
from src.data_loader import PROCESSED_DIR    
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [11]:
X_train = pd.read_parquet(PROCESSED_DIR / "X_train.parquet")   
X_test  = pd.read_parquet(PROCESSED_DIR / "X_test.parquet")    
print(X_train.shape, X_test.shape)         

(69613, 44) (29727, 44)


In [12]:
# columns that are not numeric
text_cols = X_train.select_dtypes(exclude="number").columns       
print(X_train[text_cols].nunique().sort_values(ascending=False))  

diag_3                      748
diag_2                      697
diag_1                      683
medical_specialty            70
payer_code                   18
race                          6
glimepiride                   4
glyburide-metformin           4
insulin                       4
rosiglitazone                 4
pioglitazone                  4
glyburide                     4
glipizide                     4
chlorpropamide                4
nateglinide                   4
repaglinide                   4
metformin                     4
A1Cresult                     4
max_glu_serum                 4
acarbose                      3
tolazamide                    3
acetohexamide                 2
tolbutamide                   2
miglitol                      2
troglitazone                  2
glipizide-metformin           2
metformin-rosiglitazone       2
metformin-pioglitazone        2
glimepiride-pioglitazone      1
dtype: int64


In [13]:
glu_map = {"None": 0, "Norm": 1, ">200": 2, ">300": 3}   
a1c_map = {"None": 0, "Norm": 1, ">7": 2, ">8": 3}       

def encode_labs(df):
    df = df.copy()                                       
    df["max_glu_serum"] = df["max_glu_serum"].map(glu_map)   # replace each string by its number
    df["A1Cresult"]     = df["A1Cresult"].map(a1c_map)
    return df

In [14]:
med_cols = ["metformin","repaglinide","nateglinide","chlorpropamide","glimepiride","acetohexamide",
            "glipizide","glyburide","tolbutamide","pioglitazone","rosiglitazone","acarbose","miglitol",
            "troglitazone","tolazamide","insulin","glyburide-metformin","glipizide-metformin",
            "glimepiride-pioglitazone","metformin-rosiglitazone","metformin-pioglitazone"]
med_map = {"No": 0, "Down": 1, "Steady": 2, "Up": 3}    

# apply the map to every medication column

def encode_meds(df):
    df = df.copy()
    df[med_cols] = df[med_cols].apply(lambda col: col.map(med_map))   
    return df

In [15]:
# `diag_1/2/3` hold ~700 different codes. Making them categorize into 10 categories - ref paper from Dataset
def icd9_group(code):
    if code == "Unknown":                                 
        return "Unknown"
    if code.startswith(("V", "E")):                       
        return "Other"
    n = float(code)                                       
    if 250 <= n < 251:  return "Diabetes"                
    if 390 <= n <= 459 or n == 785: return "Circulatory"  
    if 460 <= n <= 519 or n == 786: return "Respiratory"
    if 520 <= n <= 579 or n == 787: return "Digestive"
    if 580 <= n <= 629 or n == 788: return "Genitourinary"
    if 140 <= n <= 239: return "Neoplasms"                
    if 710 <= n <= 739: return "Musculoskeletal"
    if 800 <= n <= 999: return "Injury"
    return "Other"                                       

def group_diagnoses(df):
    df = df.copy()
    for c in ["diag_1", "diag_2", "diag_3"]:
        df[c] = df[c].astype(str).map(icd9_group)         
    return df

In [16]:
print(icd9_group("250.83"), icd9_group("428"), icd9_group("V57"), icd9_group("Unknown"))   

Diabetes Circulatory Other Unknown


In [17]:
prep=Pipeline([
    ("labs", FunctionTransformer(encode_labs)),
    ("meds", FunctionTransformer(encode_meds)),
    ("diags", FunctionTransformer(group_diagnoses)),
])
X_train_p = prep.fit_transform(X_train)
print(X_train_p.shape)
#X_test_p can be done later with pipeline

(69613, 44)


In [20]:
onehot_cols = ["race", "payer_code", "medical_specialty", "diag_1", "diag_2", "diag_3",
               "admission_type_id", "discharge_disposition_id", "admission_source_id"]   

ct = ColumnTransformer(
    transformers=[("onehot",
                   OneHotEncoder(min_frequency=100, handle_unknown="infrequent_if_exist", sparse_output=False),
                   onehot_cols)],        
    remainder="passthrough",             
    verbose_feature_names_out=False,     
)
ct.set_output(transform="pandas")       

X_train_enc = ct.fit_transform(X_train_p)  
#X_test_enc can be done later

# getting all processing ordinal,scalar-only pass continous numerical columns and one-hot encoding into one pipeline

In [21]:
print("Non-numeric columns left:", X_train_enc.select_dtypes(exclude="number").shape[1])    # must be 0
print(X_train_enc.filter(like="race_").head(3))                                             # look at one-hot output

Non-numeric columns left: 0
              race_AfricanAmerican  race_Asian  ...  race_Other  race_Unknown
encounter_id                                    ...                          
149190                         0.0         0.0  ...         0.0           0.0
64410                          1.0         0.0  ...         0.0           0.0
500364                         0.0         0.0  ...         0.0           0.0

[3 rows x 6 columns]


## 7. Save for the next notebooks

In [23]:
X_train_enc.to_parquet(PROCESSED_DIR / "X_train_enc.parquet")   # input for baseline + PCA
